## Predict all datapoints using the models created through training and add their outputs (and probabilities) to the existing excel sheet

In [113]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import pickle
import seaborn as sns
import pandas as pd
import numpy as np

from binarize_grades import binarize_grades

In [114]:
### Bone to predict:
BONE = 'tibia'

### Change to older model that does not record the CT_PO:
OLDER_MACHINE = True

### Use the weighted model:
USE_WEIGHTED = False

### Add mechanography features:
ADD_MECHANOGRAPHY = False

### Remove Miki
REMOVE_MIKI = True

### Use the clustering method:
USE_CLUSTERING = True

In [115]:
raw_motion_data = pd.read_excel("data/Motion full matched_for_olga.xlsx", header=1)
motion_data = raw_motion_data.replace('-', np.nan)

/tmp/ipykernel_2194753/313659671.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  motion_data = raw_motion_data.replace('-', np.nan)


In [116]:
grader_column = f'MG_{BONE}_rounded mean'

if REMOVE_MIKI:
    grader_column = f'MG_{BONE}_rounded mean no Miki'
    # add column that combines the two other graders via a mean
    graders = ['Alex', 'Felix']
    if BONE == 'tibia':
        graders = [f"{x}.1" for x in graders]
    motion_data[grader_column] = motion_data[graders].mean(axis=1)
    motion_data[grader_column] = motion_data[grader_column].round()
    motion_data[grader_column] = motion_data[grader_column].astype(int)

In [117]:
radius_columns = ['Tt.BMD', 'Tt.Ar', 'Tb.BMD', 'BV/TV', 'Tb.N', 'Tb.Th', 'Tb.Sp', 'Tb.1/N.SD', 'Tb.Ar', 'Ct.BMD', 'Ct.Th', 'Ct.Po', 'Ct.Po.Dm', 'Ct.Pm', 'Ct.Ar']
mechanography_columns = ['GRF', 'f max', 'f max rel', 'CRT tpr', 'PLEO', 'PLEC', 'EAEO', 'EAEC', 'VREO', 'VREC']
remove_features = []

radius_columns = [x for x in radius_columns if x not in remove_features]

if OLDER_MACHINE:
    # remove all rows where Ct.Po and Ct.Po.Dm are present
    motion_data = motion_data[motion_data['Ct.Po'].isna()]
    radius_columns.remove('Ct.Po')
    radius_columns.remove('Ct.Po.Dm')

tibia_columns = [f"{x}.1" for x in radius_columns]

if ADD_MECHANOGRAPHY:
    radius_columns += mechanography_columns
    tibia_columns += mechanography_columns

motion_data[radius_columns] = motion_data[radius_columns].astype(float)
motion_data[tibia_columns] = motion_data[tibia_columns].astype(float)
motion_data[tibia_columns].head()

# drop NaN rows
motion_data = motion_data.dropna(subset=radius_columns + tibia_columns)

# print the counts of each class
print(motion_data[grader_column].value_counts())
print(f"Total: {len(motion_data)}")

MG_tibia_rounded mean no Miki
2    247
1    192
4     40
3     17
5      4
Name: count, dtype: int64
Total: 500


In [118]:
motion_data_with_grader = motion_data.copy()

if not USE_CLUSTERING:
    # Method 1 of binarizing

    # binarize the labels by combining grade 1,2,3 and 4,5
    motion_data[grader_column] = motion_data[grader_column].replace({1: 0, 2: 0, 3: 0, 4: 1, 5: 1})
    print(motion_data[grader_column].value_counts())
    grade_col_name = grader_column
else:
    # Method 2 of binarizing
    # apply clustering the grade 3
    grades = motion_data[grader_column]
    motion_data = motion_data[radius_columns] if BONE == 'radius' else motion_data[tibia_columns]
    motion_data = binarize_grades(motion_data, grades)
    print(motion_data['cluster'].value_counts())
    grade_col_name = 'cluster'

Cluster 0: 439
Cluster 1: 44
Unassigned: 17
cluster
0    444
1     56
Name: count, dtype: int64


In [119]:
# put the radius columns in a np array
X = motion_data[radius_columns].values if BONE == 'radius' else motion_data[tibia_columns].values
y = motion_data[grade_col_name].values
print(X.shape, y.shape)

(500, 13) (500,)


In [120]:
scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.3, random_state=42)
train_test_split = ['train' if i in train_idx else 'test' for i in indices]

scaler.fit_transform(X[train_idx])

X_scaled = scaler.transform(X)

In [121]:
# Load the model
MACHINE = 'old' if OLDER_MACHINE else 'new'
WEIGHTED = 'balanced_' if USE_WEIGHTED else ''
print(f"Loading model: models/{BONE}_{MACHINE}_{WEIGHTED}model.pkl")

model = pickle.load(open(f"models/{BONE}_{MACHINE}_{WEIGHTED}model.pkl", 'rb'))

Loading model: models/tibia_old_model.pkl


In [122]:
# predict the labels
y_pred = model.predict_proba(X_scaled)
logits = model.decision_function(X_scaled)

final_pred = [1 if x[1] > 0.5 else 0 for x in y_pred]
final_probas = [x[1] if x[1] > 0.5 else 1 - x[1] for x in y_pred]

predictions = pd.DataFrame(columns=['true', 'pred', 'probability', 'correct'], index=motion_data_with_grader.index)
predictions['true'] = ['pass' if x == 0 else 'fail' for x in y]
predictions['pred'] = ['pass' if x == 0 else 'fail' for x in final_pred]
predictions['probability'] = final_probas
predictions['correct'] = predictions['true'] == predictions['pred']
predictions['original_set'] = train_test_split
predictions['grade'] = motion_data_with_grader[grader_column]
predictions['logits'] = logits
predictions

,true,pred,probability,correct,original_set,grade,logits
0,pass,pass,0.918338,True,test,2,-2.419978
1,pass,pass,0.674995,True,train,2,-0.730864
2,fail,fail,0.845061,True,test,3,1.696375
3,pass,pass,0.908263,True,train,2,-2.292610
4,pass,pass,0.988765,True,train,1,-4.477431
...,...,...,...,...,...,...,...
495,pass,pass,0.615795,True,test,2,-0.471736
496,pass,pass,0.971926,True,train,2,-3.544441
497,fail,pass,0.692974,False,test,3,-0.814058
498,pass,pass,0.932912,True,train,1,-2.632303


In [124]:
# write to csv
predictions.to_csv(f"predictions/{BONE}_{MACHINE}_{WEIGHTED}predictions.csv", index=False)

## Check if ordering is correct

In [ ]:
motion_data_with_grader

In [126]:
# check if in the raw data every grade 1, 2 corresponds to a pass and 4, 5 to a fail in the true labels of the predictions df
if BONE == 'tibia':
    grades = ['MG_tibia_rounded mean no Miki', 'MG_tibia_rounded mean']
else:
    grades = ['MG_radius_rounded mean no Miki', 'MG_radius_rounded mean']

c = 0


for grade in grades:
    print(f"Checking if {grade} corresponds to the predictions")
    for idx, row in motion_data_with_grader.iterrows():
        if row[grade] == 1 or row[grade] == 2:
            if predictions.loc[idx]['true'] != 'pass':
                print(f"Grade {row[grade]} does not correspond to pass")
                c += 1
        elif row[grade] == 4 or row[grade] == 5:
            if predictions.loc[idx]['true'] != 'fail':
                print(f"Grade {row[grade]} does not correspond to fail")
                c += 1

Checking if MG_tibia_rounded mean no Miki corresponds to the predictions
Checking if MG_tibia_rounded mean corresponds to the predictions
